In [2]:
# Load env variables and create client
from dotenv import load_dotenv
from anthropic import Anthropic


load_dotenv()

client = Anthropic()
model = "claude-sonnet-4-5"

In [3]:
# Helper functions


def add_user_message(messages, message):
    if isinstance(message, list):
        user_message = {
            "role": "user",
            "content": message,
        }
    else:
        user_message = {
            "role": "user",
            "content": [{"type": "text", "text": message}],
        }
    messages.append(user_message)


def add_assistant_message(messages, message):
    if isinstance(message, list):
        assistant_message = {
            "role": "assistant",
            "content": message,
        }
    elif hasattr(message, "content"):
        content_list = []
        for block in message.content:
            if block.type == "text":
                content_list.append({"type": "text", "text": block.text})
            elif block.type == "tool_use":
                content_list.append(
                    {
                        "type": "tool_use",
                        "id": block.id,
                        "name": block.name,
                        "input": block.input,
                    }
                )
        assistant_message = {
            "role": "assistant",
            "content": content_list,
        }
    else:
        # String messages need to be wrapped in a list with text block
        assistant_message = {
            "role": "assistant",
            "content": [{"type": "text", "text": message}],
        }
    messages.append(assistant_message)


def chat_stream(
    messages,
    system=None,
    temperature=1.0,
    stop_sequences=[],
    tools=None,
    tool_choice=None,
    betas=[],
):
    params = {
        "model": model,
        "max_tokens": 1000,
        "messages": messages,
        "temperature": temperature,
        "stop_sequences": stop_sequences,
    }

    if tool_choice:
        params["tool_choice"] = tool_choice

    if tools:
        params["tools"] = tools

    if system:
        params["system"] = system

    if betas:
        params["betas"] = betas

    return client.beta.messages.stream(**params)


def text_from_message(message):
    return "\n".join([block.text for block in message.content if block.type == "text"])

In [4]:
# Tool definition
from anthropic.types import ToolParam

save_article_schema = ToolParam(
    {
        "name": "save_article",
        "description": "Saves a scholarly journal article",
        "input_schema": {
            "type": "object",
            "properties": {
                "abstract": {
                    "type": "string",
                    "description": "Abstract of the article. One short sentence max",
                },
                "meta": {
                    "type": "object",
                    "properties": {
                        "word_count": {
                            "type": "integer",
                            "description": "Word count",
                        },
                        "review": {
                            "type": "string",
                            "description": "Eight sentence review of the paper",
                        },
                    },
                    "required": ["word_count", "review"],
                },
            },
            "required": ["abstract", "meta"],
        },
    }
)
save_short_article_schema = ToolParam(
    {
        "name": "save_article",
        "description": "Saves a scholarly journal article",
        "input_schema": {
            "type": "object",
            "properties": {
                "abstract": {
                    "type": "string",
                    "description": "Abstract of the article. One short sentence max",
                },
                "meta": {
                    "type": "object",
                    "properties": {
                        "word_count": {
                            "type": "integer",
                            "description": "Word count",
                        },
                        "review": {
                            "type": "string",
                            "description": "Review of paper. One short sentence max",
                        },
                    },
                    "required": ["word_count", "review"],
                },
            },
            "required": ["abstract", "meta"],
        },
    }
)


def save_article(**kwargs):
    return "Article saved!"


In [5]:
# Tool Running
import json


def run_tool(tool_name, tool_input):
    if tool_name == "save_article":
        return save_article(**tool_input)


def run_tools(message):
    tool_requests = [block for block in message.content if block.type == "tool_use"]
    tool_result_blocks = []

    for tool_request in tool_requests:
        try:
            tool_output = run_tool(tool_request.name, tool_request.input)
            tool_result_block = {
                "type": "tool_result",
                "tool_use_id": tool_request.id,
                "content": json.dumps(tool_output),
                "is_error": False,
            }
        except Exception as e:
            tool_result_block = {
                "type": "tool_result",
                "tool_use_id": tool_request.id,
                "content": f"Error: {e}",
                "is_error": True,
            }

        tool_result_blocks.append(tool_result_block)

    return tool_result_blocks

In [6]:
# Run conversation
def run_conversation(messages, tools=[], tool_choice=None, fine_grained=False):
    while True:
        with chat_stream(
            messages,
            tools=tools,
            betas=["fine-grained-tool-streaming-2025-05-14"] if fine_grained else [],
            tool_choice=tool_choice,
        ) as stream:
            for chunk in stream:
                if chunk.type == "text":
                    print(chunk.text, end="")

                if chunk.type == "content_block_start":
                    if chunk.content_block.type == "tool_use":
                        print(f'\n>>> Tool Call: "{chunk.content_block.name}"')

                if chunk.type == "input_json" and chunk.partial_json:
                    print(chunk.partial_json, end="")

                if chunk.type == "content_block_stop":
                    print("\n")

            response = stream.get_final_message()

        add_assistant_message(messages, response)

        if response.stop_reason != "tool_use":
            break

        tool_results = run_tools(response)
        add_user_message(messages, tool_results)

        if tool_choice:
            break

    return messages

In [7]:
messages = []

add_user_message(
    messages,
    "Create and save a fake computer science article",
)

run_conversation(
    messages,
    tools=[save_article_schema],
)

I'll create and save a fake computer science article for you.


>>> Tool Call: "save_article"
{"abstract": "This paper presents a novel quantum-inspired algorithm for optimizing neural network architectures using adaptive gradient descent in distributed computing environments.", "meta": {"word_count":8450,"review":"This paper introduces an innovative approach to automated neural architecture search by leveraging quantum computing principles. The authors propose a hybrid algorithm that combines classical gradient descent with quantum annealing techniques to explore the architecture search space more efficiently. Experimental results demonstrate a 34% reduction in training time compared to state-of-the-art methods on standard benchmarks. The mathematical framework is rigorous and well-presented, though some proofs in the appendix could benefit from additional clarity. The distributed implementation shows excellent scalability up to 128 nodes. However, the paper lacks discussion of the al

[{'role': 'user',
  'content': [{'type': 'text',
    'text': 'Create and save a fake computer science article'}]},
 {'role': 'assistant',
  'content': [{'type': 'text',
    'text': "I'll create and save a fake computer science article for you."},
   {'type': 'tool_use',
    'id': 'toolu_0156qNUdERSd3Wts9tDCz1An',
    'name': 'save_article',
    'input': {'abstract': 'This paper presents a novel quantum-inspired algorithm for optimizing neural network architectures using adaptive gradient descent in distributed computing environments.',
     'meta': {'word_count': 8450,
      'review': "This paper introduces an innovative approach to automated neural architecture search by leveraging quantum computing principles. The authors propose a hybrid algorithm that combines classical gradient descent with quantum annealing techniques to explore the architecture search space more efficiently. Experimental results demonstrate a 34% reduction in training time compared to state-of-the-art methods on

I'll create and save a fake computer science article for you.


>>> Tool Call: "save_article"
{"abstract": "This paper introduces a novel quantum-inspired algorithm for optimizing neural network architectures using adaptive graph traversal techniques.", "meta": {"word_count":4782,"review":"This paper presents an innovative approach to neural architecture search by combining principles from quantum computing with classical graph theory. The authors propose a quantum-inspired traversal method that explores the architecture search space more efficiently than traditional methods. The experimental results demonstrate significant improvements in both search time and model performance across standard benchmarks. However, the scalability of the approach to very large search spaces remains questionable. The theoretical foundations are sound, though some proofs could be more rigorous. The paper makes valuable contributions to automated machine learning research. Overall, this is a solid piece of work that advances the state-of-the-art in neural architecture optimization. Future work should focus on addressing the computational overhead in distributed settings."}}

Done! I've created and saved a fake computer science article about a quantum-inspired algorithm for neural network architecture optimization. The article includes:

- **Abstract**: A brief description of a novel quantum-inspired algorithm for optimizing neural networks
- **Word count**: 4,782 words
- **Review**: An eight-sentence peer review covering the paper's contributions, strengths, weaknesses, and future directions

The article has been successfully saved!


In [8]:
messages = []

add_user_message(
    messages,
    "Create and save a fake computer science article",
)

run_conversation(
    messages,
    tools=[save_article_schema],
    fine_grained=True
)

I'll create and save a fake computer science article for you.


>>> Tool Call: "save_article"
{"abstract": "This paper presents a novel quantum-resistant blockchain consensus algorithm that achieves O(log n) time complexity while maintaining Byzantine fault tolerance.", "meta": {
  "word_count": 4782,
  "review": "This paper introduces QuantumChain, an innovative consensus mechanism designed to address the growing threat of quantum computing to existing blockchain systems. The authors propose a hybrid approach combining lattice-based cryptography with a modified proof-of-stake protocol that reduces computational overhead significantly. The theoretical analysis demonstrates impressive time complexity improvements over traditional Byzantine fault-tolerant algorithms. Experimental results on a network of 10,000 nodes show 40% faster transaction finality compared to existing solutions. The quantum-resistance claims are well-supported by rigorous security proofs based on the Learning with E

[{'role': 'user',
  'content': [{'type': 'text',
    'text': 'Create and save a fake computer science article'}]},
 {'role': 'assistant',
  'content': [{'type': 'text',
    'text': "I'll create and save a fake computer science article for you."},
   {'type': 'tool_use',
    'id': 'toolu_01976CPpGSHzapUrcBWHzuxn',
    'name': 'save_article',
    'input': {'abstract': 'This paper presents a novel quantum-resistant blockchain consensus algorithm that achieves O(log n) time complexity while maintaining Byzantine fault tolerance.',
     'meta': {'word_count': 4782,
      'review': 'This paper introduces QuantumChain, an innovative consensus mechanism designed to address the growing threat of quantum computing to existing blockchain systems. The authors propose a hybrid approach combining lattice-based cryptography with a modified proof-of-stake protocol that reduces computational overhead significantly. The theoretical analysis demonstrates impressive time complexity improvements over tradi